In [13]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [14]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
import pandas as pd

books = pd.read_csv("../data/books_cleaned.csv")

In [16]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description
0,9781804510407,Other Side of the Wire,Ralph J. Whitehead,"Germany. Heer;Germany. Heer. Reserve Corps, 14...","""By focusing on one of the principal German fo...",NaN,2022,NaN,NaN,Other Side of the Wire: Volume 2 - the Battle ...,"9781804510407 ""By focusing on one of the princ..."
1,9781108663229,African Literature and the CIA,Caroline Davis,Written communication;African literature;CIA;U...,"""During the period of decolonisation in Africa...",NaN,2020,NaN,NaN,African Literature and the CIA: Networks of Au...,"9781108663229 ""During the period of decolonisa..."
2,9780367418489,Interior Provocations,Anca I. Lasc;Pratt Institute Staff,Architecture;Interior architecture;Congresses;...,"""Interior Provocations: History, Theory, and P...",NaN,2020,NaN,NaN,Interior Provocations,"9780367418489 ""Interior Provocations: History,..."
3,9781800326514,An Unfortunate Christmas Murder,Hannah Hendy,English literature,"‘Tis the season for gold, frankincense and mur...",https://covers.openlibrary.org/b/id/15179740-L...,2022,NaN,NaN,An Unfortunate Christmas Murder,"9781800326514 ‘Tis the season for gold, franki..."
4,9781108485852,Legitimacy of Unseen Actors in International A...,Freya Baetens,Arbitration (international law);Jurisdiction (...,"""'Unseen actors' are vital to the functioning ...",NaN,2019,NaN,496.0,Legitimacy of Unseen Actors in International A...,"9781108485852 ""'Unseen actors' are vital to th..."
...,...,...,...,...,...,...,...,...,...,...,...
4995,9781952223174,"Why, As a Muslim, I Support Liberty",Mustafa Akyol,Religion;Philosophy;Liberty;Islam;Muslims;Reli...,"Islam, the second largest religion in the worl...",NaN,2021,NaN,192.0,"Why, As a Muslim, I Support Liberty","9781952223174 Islam, the second largest religi..."
4996,9781532147555,The Horror at Happy Landings,Robert Lawrence Stine;Kelly Matthews;Nichole M...,NaN,Two Martians unexpectedly land on Earth and ha...,https://covers.openlibrary.org/b/id/14408331-L...,2020,NaN,NaN,The Horror at Happy Landings: Just Beyond Volu...,9781532147555 Two Martians unexpectedly land o...
4997,9789004278783,The Blinded State,Mitko B. Panov,"Byzantine empire, history;Macedonia, history;H...","""This book is a revisionist account of Samuel'...",NaN,2019,NaN,478.0,The Blinded State,"9789004278783 ""This book is a revisionist acco..."
4998,9781784753474,Walls,Hollie Overton,"Abused women;Murder;Fiction;Fiction, suspense;...","""For fans of The Girl on the Train, The Walls ...",NaN,2018,NaN,416.0,Walls,"9781784753474 ""For fans of The Girl on the Tra..."


In [17]:
books["tagged_description"].to_csv("../data/tagged_description.txt",
                                   sep = "\n",
                                   index = False,
                                   header = False)

In [18]:
raw_documents = TextLoader("../data/tagged_description.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=0.1, chunk_overlap=0, separator="\n")
# chunk_size = 0.1 -- if this is more than one it may split on a chunk size rather than the separator
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 416, which is longer than the specified 0
Created a chunk of size 1035, which is longer than the specified 0
Created a chunk of size 767, which is longer than the specified 0
Created a chunk of size 951, which is longer than the specified 0
Created a chunk of size 1808, which is longer than the specified 0
Created a chunk of size 413, which is longer than the specified 0
Created a chunk of size 308, which is longer than the specified 0
Created a chunk of size 1182, which is longer than the specified 0
Created a chunk of size 889, which is longer than the specified 0
Created a chunk of size 1745, which is longer than the specified 0
Created a chunk of size 389, which is longer than the specified 0
Created a chunk of size 983, which is longer than the specified 0
Created a chunk of size 619, which is longer than the specified 0
Created a chunk of size 1261, which is longer than the specified 0
Created a chunk of size 937, which is longer than the specified 0
Creat

In [19]:
documents[0]

Document(metadata={'source': '../data/tagged_description.txt'}, page_content='"9781804510407 ""By focusing on one of the principal German formations involved in the Somme fighting, the author brings to life this little-known period, from the initial German advance on the Somme in September 1914 through the formation of the front that became so well known two years later. Covers the early fighting around villages such as Serre, Beaumont-Hamel, Thiepval, Ovillers, La Boisselle and Fricourt."')

In [20]:
db_books = Chroma.from_documents(
    documents,
    embedding=OpenAIEmbeddings())

In [21]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 3)
docs

[Document(id='88dc7115-79e4-4bf5-b60c-90fe39fb2906', metadata={'source': '../data/tagged_description.txt'}, page_content='9781426331565 Packed with colorful photographs of adorable animals braving the elements, this picture book for preschoolers introduces kids to the weather they experience every day, including rain, clouds, sunshine, snow, storms, and more.'),
 Document(id='c5e43c8c-9a22-472c-93f4-2b6209836a3f', metadata={'source': '../data/tagged_description.txt'}, page_content='9780593462034 In a village on the African plains, a little girl stalls bedtime by saying good night to various animals and objects.'),
 Document(id='14228400-fe9d-4d58-b350-8b9654af98d6', metadata={'source': '../data/tagged_description.txt'}, page_content='9781663961853 Who Grows Up Here? - Beautiful drawings and lyrical text take the reader from the tundra to the tropics to discover a whole new world of baby animals.')]

In [22]:
# get the full data of the top book from the query
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description


In [23]:
def get_semantic_recommendation(query: str) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 30)

    books_list = []

    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)]

In [27]:
get_semantic_recommendation("A book about a woman astronaut")

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description
830,9781328948939,Brightly burning,Alexa Donne,Orphans;Fiction;Artificial intelligence;Virus ...,Seventeen-year-old Stella Ainsley wants just o...,NaN,2018,NaN,394.0,Brightly burning,9781328948939 Seventeen-year-old Stella Ainsle...
953,9781338262513,Satellite space mission,AnnMarie Anderson,Juvenile fiction;Artificial satellites;Explora...,Space Camp is the launching pad to becoming an...,NaN,2018,NaN,91.0,Satellite space mission,9781338262513 Space Camp is the launching pad ...
1462,9781421596976,Astra lost in space,篠原 健太,Space ships;Exploration;Astronauts;Planets;Int...,"After crashing on planet Icriss, all seems los...",NaN,2018,NaN,NaN,Astra lost in space: Revelation,"9781421596976 After crashing on planet Icriss,..."
2365,9781446432402,Beegu,Alexis Deacon,Extraterrestrial beings;Fiction;Children's fic...,A small creature from space finds no welcome o...,NaN,2018,NaN,NaN,Beegu,9781446432402 A small creature from space find...
2639,9798227464255,The Antarctica Conspiracy,"Henry, Lisa",Series:Time-to-Orbit-Unknown;Science Fiction;M...,"It hasn't been easy, but under the careful han...",https://covers.openlibrary.org/b/id/15128146-L...,2024,NaN,716.0,The Antarctica Conspiracy,"9798227464255 It hasn't been easy, but under t..."
3233,9780262539944,The Curie Society,Heather Einhorn;Adam Staffaroni;Janet Harvey;J...,"Secret societies;Comic books, strips;Women;Soc...","""An action-adventure original graphic novel, f...",https://covers.openlibrary.org/b/id/10874209-L...,2021,NaN,168.0,The Curie Society,"9780262539944 ""An action-adventure original gr..."
3437,9781642595826,Eslanda Second Ed,Barbara Ransby,African American anthropologists;Biography;Wom...,Chronicles the eventful life of the anthropolo...,NaN,2022,NaN,422.0,Eslanda Second Ed: The Large and Unconventiona...,9781642595826 Chronicles the eventful life of ...
4239,9780356521435,Revenant-X,David Wellington,horror;Science fiction;Sci Fi;Outer Space,"""The crew of the Artemis - led by Firewatch ag...",https://covers.openlibrary.org/b/id/14833503-L...,2024,NaN,500.0,Revenant-X,"9780356521435 ""The crew of the Artemis - led b..."
4500,9780316307598,Robot rescue,Drew Brockington,Juvenile fiction;Graphic novels;Robots;Astrona...,When Cat-Stro-Bot gets into trouble while help...,https://covers.openlibrary.org/b/id/8996789-L.jpg,2018,NaN,NaN,Robot rescue,9780316307598 When Cat-Stro-Bot gets into trou...
4546,9781465477132,Universe,Giles Sparrow,Juvenile literature;Galaxies;Cosmology;Cosmolo...,"Children will find out about the universe, our...",NaN,2018,NaN,64.0,Universe,9781465477132 Children will find out about the...
